In [1]:
from rdkit.Chem import AllChem
from numpy import zeros
from rdkit import DataStructs
from rdkit.Chem import MolFromSmiles
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, balanced_accuracy_score, matthews_corrcoef, f1_score
from pandas import DataFrame, concat
from pickle import load

In [2]:
def calc_morgan(mols):
    """ генерация молекулярных отпечатков по методу Моргана с радиусом 2 и длиной 2048
    """
    for_df = []
    fpgen = AllChem.GetMorganGenerator(radius=2, )
    for m in mols:
        arr = zeros((1,), dtype=int)
        DataStructs.ConvertToNumpyArray(fpgen.GetFingerprint(m), arr)
        for_df.append(arr)
    return DataFrame(for_df)

with open('../data/classifier/modeling_data.pickle', 'rb') as file:
    data = load(file)

In [3]:
molecules = [
    MolFromSmiles(mol) for mol in data['SMILES']
]

In [4]:
X = calc_morgan(molecules)
Y = data['Activity']
XY = concat((X, Y), axis=1)
XY

,0,1,2,3,4,5,6,7,8,9,...,2039,2040,2041,2042,2043,2044,2045,2046,2047,Activity
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75924,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
75925,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
75926,0,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
75927,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [6]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.20)

In [ ]:
gb = HistGradientBoostingClassifier(class_weight='balanced')
pg = {'learning_rate': [0.01, 0.1, 1],
      'max_depth': [None, 1, 5, 7, 10],
      'max_iter': [100, 200, 300],
      'max_leaf_nodes': [30, 40, 50],
      'min_samples_leaf': [10, 20, 30],
      'l2_regularization': [0, 0.3, 0.6]
      }
cv = RepeatedKFold(n_splits=5, n_repeats=5)
gs = GridSearchCV(gb, pg, verbose=3, cv=cv, refit='f1', scoring=('f1', 'balanced_accuracy'))

gs.fit(x_train, y_train)

Fitting 25 folds for each of 1215 candidates, totalling 30375 fits
[CV 1/25] END l2_regularization=0, learning_rate=0.01, max_depth=None, max_iter=100, max_leaf_nodes=30, min_samples_leaf=10; balanced_accuracy: (test=0.777) f1: (test=0.869) total time= 5.0min
[CV 2/25] END l2_regularization=0, learning_rate=0.01, max_depth=None, max_iter=100, max_leaf_nodes=30, min_samples_leaf=10; balanced_accuracy: (test=0.777) f1: (test=0.865) total time= 6.6min
[CV 3/25] END l2_regularization=0, learning_rate=0.01, max_depth=None, max_iter=100, max_leaf_nodes=30, min_samples_leaf=10; balanced_accuracy: (test=0.779) f1: (test=0.871) total time= 5.3min
[CV 4/25] END l2_regularization=0, learning_rate=0.01, max_depth=None, max_iter=100, max_leaf_nodes=30, min_samples_leaf=10; balanced_accuracy: (test=0.777) f1: (test=0.870) total time= 6.2min
[CV 5/25] END l2_regularization=0, learning_rate=0.01, max_depth=None, max_iter=100, max_leaf_nodes=30, min_samples_leaf=10; balanced_accuracy: (test=0.782) f1: 

In [ ]:
gs.best_estimator_

In [ ]:
y_pred = gs.predict(x_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel().tolist()
precision = tp / (tp + fp)
recall = tp / (tp + fn)

In [ ]:
print(f'ACC = {accuracy_score(y_test, y_pred):.3f}')
print(f'BA = {balanced_accuracy_score(y_test, y_pred):.3f}')
print(f'MCC = {matthews_corrcoef(y_test, y_pred):.3f}')
print(f'ROC AUC = {roc_auc_score(y_test, y_pred):.3f}')
print(f'F1 = {f1_score(y_test, y_pred):.3f}')
print(f'Precision = {precision:.3f}')
print(f'Recall = {recall:.3f}')